# MSc Final Project: Visible In-Progress (VIP) notebook

| | |
|---|---|
| **Project title** | *Multimodal Brain Tumour Classification Using MRI and CT Images* |
| **Student name / number** | *Niharika Rayavarapu / 24175430* |
| **Supervisor** | *Wajih Islam* |
| **GitHub repository** | *https://github.com/Niharika1003/msc-project-rayavarapu* |
| **Overleaf project** | *link to your IPR/FPR report* |
| **Ethics approval** | *Not required (reason) / Approved: ethics number* |

This notebook is your **practical work**: data, artefact, experiments and results. Your **report** (IPR 10%, FPR 70%) is written in the Overleaf template. Your **demo/viva (20%)** is run from this notebook.


1. **Add your supervisor as a collaborator** on your GitHub repository in week 1 (Settings → Collaborators → Add people). Your supervisor cannot review your work without this access.
2. **Commit and push after every task, after every change, and whenever you change an existing solution.** Your history shows how your project developed.
```bash
git status
git add .
git commit -m "CODE: what you did"
git push
```
3. Start each commit message with the part you worked on: `DATA:` · `CODE:` · `EXP:` · `TEST:` · `RESULT:` · `FIX:`
   e.g. `FIX: changed learning rate from 0.01 to 0.001, accuracy up to 0.87`
4. **Develop at least two parts of your artefact on their own branches** (for example your baseline and your proposed method). See *Working on a branch* below.
5. **Before each supervision meeting**, push your latest work. Your supervisor reviews what changed.
6. Never commit personal data or data that needs ethics approval. Keep the repository private, and do not rewrite its history.

## Working on a branch

A branch lets you build one part of your project without breaking the working version on `main`. Develop **at least two parts** this way, e.g. `baseline` and `proposed-method`.

**1. Create a branch and work on it**
```bash
git checkout -b baseline
```
Commit as usual while you work:
```bash
git status
git add .
git commit -m "CODE: baseline model, first version"
git push -u origin baseline
```
(`-u origin baseline` is needed only the first time you push a new branch.)

**2. When the part works, merge it into `main`**
```bash
git checkout main
git merge baseline
git push
```

**3. Start the next part on a new branch**
```bash
git checkout -b proposed-method
```

Check which branch you are on with `git branch` (the current one has a `*`). On GitHub, your supervisor can see every branch and its commits.

# Imports

In [1]:
# Essential libraries
import os
from pathlib import Path
# Data handling
import numpy as np
import pandas as pd
# Medical image processing
import nibabel as nib
# Visualisation
import matplotlib.pyplot as plt
# Machine learning
import sklearn
import tensorflow as tf
# Display settings
pd.set_option("display.max_columns", None)
print("All libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

All libraries imported successfully!
TensorFlow version: 2.21.0


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: project set-up, folders and environment"
git push
```

# Research question and objectives

**Problem:** *Brain tumour classification using medical images can be challenging because different tumour types may have different imaging characteristics. MRI and CT provide different types of information about the brain.*

**Main research question:** *How effectively can deep learning techniques classify different types of brain tumours using MRI and CT images, individually and in combination?*

**Sub-questions:**
*1.How can MRI and CT images be preprocessed for deep learning classification?*

*2.How effectively can a 2D CNN classify brain tumour types using MRI images?*

*3.How does classification performance differ between MRI-only and CT-only models?*

*4.Can combining MRI and CT improve classification performance?*

**Objectives (specific and measurable):**
*1.Explore and understand a paired MRI–CT brain imaging dataset.*

*2.Develop a consistent preprocessing pipeline for MRI and CT images.*

*3.Develop a baseline 2D CNN for MRI-based tumour classification.*

*4.Investigate CT-based classification.*

*5.Develop a multimodal classification approach combining MRI and CT.*

*6.Compare the models using accuracy, precision, recall and F1-score.*


# Data
*FPR: Methodology*

**Source and licence:** ...

**Ethics / GDPR:** does the data include people or personal information? How is it stored and protected?

In [2]:
groups = {
    "meningioma": r"C:\Users\nihar\Downloads\meningioma\meningioma",

    "cerebrovascular disease": r"C:\Users\nihar\Downloads\cerebrovascular-disease\cerebrovascular-disease",

    "vestibular schwannoma": r"C:\Users\nihar\Downloads\vestibular-schwannoma (1)\vestibular-schwannoma",

    "pituitary tumor": r"C:\Users\nihar\Downloads\pituitary-tumor\pituitary-tumor"
}

In [3]:
from pathlib import Path

for diagnosis, folder_path in groups.items():
    folder = Path(folder_path)
    print(diagnosis, ":", folder.exists())

meningioma : True
cerebrovascular disease : True
vestibular schwannoma : True
pituitary tumor : True


In [7]:
from pathlib import Path
import pandas as pd

records = []

for diagnosis, folder_path in groups.items():

    folder = Path(folder_path)

    for file in folder.rglob("*"):
        if file.is_file() and (
            file.name.endswith(".nii") or
            file.name.endswith(".nii.gz")
        ):

            records.append({
                "Diagnosis": diagnosis,
                "File Name": file.name,
                "File Path": str(file)
            })

# Create DataFrame
df = pd.DataFrame(records)

print("Dataset shape:", df.shape)

display(df.head(10))

print("\nFiles per diagnosis:")
display(df.groupby("Diagnosis").size().reset_index(name="Total Files"))

Dataset shape: (420, 3)


,Diagnosis,File Name,File Path
0,meningioma,CE-T1W.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
1,meningioma,CT-BONE.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
2,meningioma,CT-BRAIN.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
3,meningioma,CE-T1W.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
4,meningioma,CT-BONE.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
5,meningioma,CT-BRAIN.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
6,meningioma,CE-T1W.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
7,meningioma,CT-BONE.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
8,meningioma,CT-BRAIN.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...
9,meningioma,CE-T1W.nii.gz,C:\Users\nihar\Downloads\meningioma\meningioma...



Files per diagnosis:


,Diagnosis,Total Files
0,cerebrovascular disease,60
1,meningioma,60
2,pituitary tumor,240
3,vestibular schwannoma,60


### Data-quality check and cleaning log

| Step | Problem found | What I changed | Why |
|---|---|---|---|
| 1 | The downloaded dataset contained four categories, including cerebrovascular disease.|Identified and excluded cerebrovascular disease from the tumour classification dataset. | The project focuses on three brain tumour types, and cerebrovascular disease is not a tumour.|
| 2 | The dataset contains multiple NIfTI files for each patient.|Started checking MRI and CT files separately.| To identify the correct scans belonging to each patient.|
| 3 | MRI and CT scans need to be matched by patient.|Patient-level matching will be performed using diagnosis and patient ID. | To ensure that MRI and CT images belong to the same patient.|
| 4 | Image volumes may have different dimensions and voxel spacing.|Dimensions and voxel spacing will be checked before preprocessing. | To understand differences between scans and prepare them consistently.|

In [10]:
# cleaning steps
# Step 1: Remove cerebrovascular disease

tumour_df = df[
    df["Diagnosis"] != "cerebrovascular disease"
].copy()

# Step 2: Check remaining categories

print("Remaining tumour categories:")
print(tumour_df["Diagnosis"].unique())

# Step 3: Check total files

print("\nTotal tumour NIfTI files:", len(tumour_df))

# Step 4: Check files per category

display(
    tumour_df.groupby("Diagnosis")
    .size()
    .reset_index(name="Total Files")
)

Remaining tumour categories:
<StringArray>
['meningioma', 'vestibular schwannoma', 'pituitary tumor']
Length: 3, dtype: str

Total tumour NIfTI files: 360


,Diagnosis,Total Files
0,meningioma,60
1,pituitary tumor,240
2,vestibular schwannoma,60


In [11]:
tumour_df["Scan Type"] = tumour_df["File Name"].apply(
    lambda x: (
        "MRI" if "CE-T1W" in x
        else "CT-BONE" if "CT-BONE" in x
        else "CT-BRAIN" if "CT-BRAIN" in x
        else "Other"
    )
)

scan_summary = (
    tumour_df.groupby(["Diagnosis", "Scan Type"])
    .size()
    .reset_index(name="Count")
)

display(scan_summary)

,Diagnosis,Scan Type,Count
0,meningioma,CT-BONE,20
1,meningioma,CT-BRAIN,20
2,meningioma,MRI,20
3,pituitary tumor,CT-BONE,60
4,pituitary tumor,CT-BRAIN,60
5,pituitary tumor,MRI,120
6,vestibular schwannoma,CT-BONE,20
7,vestibular schwannoma,CT-BRAIN,20
8,vestibular schwannoma,MRI,20


In [12]:
from pathlib import Path

for diagnosis, folder_path in groups.items():

    if diagnosis == "cerebrovascular disease":
        continue

    folder = Path(folder_path)

    patient_folders = [
        p for p in folder.iterdir()
        if p.is_dir()
    ]

    print(
        diagnosis,
        "Number of patient folders:",
        len(patient_folders)
    )

meningioma Number of patient folders: 20
vestibular schwannoma Number of patient folders: 20
pituitary tumor Number of patient folders: 60


In [13]:
from pathlib import Path

pituitary_folder = Path(groups["pituitary tumor"])

# Select the first patient folder
patient_folder = sorted(
    [p for p in pituitary_folder.iterdir() if p.is_dir()]
)[0]

print("Patient folder:", patient_folder.name)
print("\nFiles inside this patient folder:")

for file in sorted(patient_folder.rglob("*")):
    if file.is_file() and (
        file.name.endswith(".nii") or
        file.name.endswith(".nii.gz")
    ):
        print(file.name)

Patient folder: sub-01

Files inside this patient folder:
CE-T1W.nii.gz
CT-BONE.nii.gz
CT-BRAIN.nii.gz
NCE-T1W.nii.gz


In [16]:
pituitary_folder = Path(groups["pituitary tumor"])

patient_summary = []

for patient in sorted(p for p in pituitary_folder.iterdir() if p.is_dir()):

    files = [
        f.name for f in patient.rglob("*")
        if f.is_file() and (
            f.name.endswith(".nii") or
            f.name.endswith(".nii.gz")
        )
    ]

    patient_summary.append({
        "Patient ID": patient.name,
        "CE-T1W MRI": "CE-T1W.nii.gz" in files,
        "NCE-T1W MRI": "NCE-T1W.nii.gz" in files,
        "CT-BONE": "CT-BONE.nii.gz" in files,
        "CT-BRAIN": "CT-BRAIN.nii.gz" in files,
        "Total files": len(files)
    })

pituitary_summary = pd.DataFrame(patient_summary)

display(pituitary_summary)
display(
    pituitary_summary.groupby(
        ["CE-T1W MRI", "NCE-T1W MRI", "CT-BONE", "CT-BRAIN"]
    ).size().reset_index(name="Number of patients")
)

,Patient ID,CE-T1W MRI,NCE-T1W MRI,CT-BONE,CT-BRAIN,Total files
0,sub-01,True,True,True,True,4
1,sub-02,True,True,True,True,4
2,sub-03,True,True,True,True,4
3,sub-04,True,True,True,True,4
4,sub-05,True,True,True,True,4
5,sub-06,True,True,True,True,4
6,sub-07,True,True,True,True,4
7,sub-08,True,True,True,True,4
8,sub-09,True,True,True,True,4
9,sub-10,True,True,True,True,4


,CE-T1W MRI,NCE-T1W MRI,CT-BONE,CT-BRAIN,Number of patients
0,True,True,True,True,60


In [18]:
for diagnosis in ["meningioma", "vestibular schwannoma"]:

    folder = Path(groups[diagnosis])

    records = []

    for patient in sorted(p for p in folder.iterdir() if p.is_dir()):

        files = [
            f.name for f in patient.rglob("*")
            if f.is_file() and f.name.endswith((".nii", ".nii.gz"))
        ]

        records.append({
            "Diagnosis": diagnosis,
            "Patient": patient.name,
            "CE-T1W": any("CE-T1W" in f for f in files),
            "NCE-T1W": any("NCE-T1W" in f for f in files),
            "CT-BONE": any("CT-BONE" in f for f in files),
            "CT-BRAIN": any("CT-BRAIN" in f for f in files)
        })

    result = pd.DataFrame(records)

    print("\n", diagnosis)
    display(
        result.groupby(
            ["CE-T1W", "NCE-T1W", "CT-BONE", "CT-BRAIN"]
        ).size().reset_index(name="Number of patients")
    )


 meningioma


,CE-T1W,NCE-T1W,CT-BONE,CT-BRAIN,Number of patients
0,True,False,True,True,20



 vestibular schwannoma


,CE-T1W,NCE-T1W,CT-BONE,CT-BRAIN,Number of patients
0,True,False,True,True,20


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "DATA: cleaned dataset, see cleaning log"
git push
```

In [22]:
from pathlib import Path
import pandas as pd

records = []

# Process only the three tumour categories
for diagnosis in [
    "meningioma",
    "pituitary tumor",
    "vestibular schwannoma"
]:

    folder = Path(groups[diagnosis])

    # Go through each patient folder
    for patient in sorted(p for p in folder.iterdir() if p.is_dir()):

        files = list(patient.rglob("*.nii*"))

        # Find the required scan paths
        scan_paths = {}

        for scan in files:
            name = scan.name

            if name.startswith("CE-T1W"):
                scan_paths["CE-T1W MRI"] = str(scan)

            elif name.startswith("NCE-T1W"):
                scan_paths["NCE-T1W MRI"] = str(scan)

            elif name.startswith("CT-BONE"):
                scan_paths["CT-BONE"] = str(scan)

            elif name.startswith("CT-BRAIN"):
                scan_paths["CT-BRAIN"] = str(scan)

        records.append({
            "Diagnosis": diagnosis,
            "Patient ID": patient.name,
            "CE-T1W MRI": scan_paths.get("CE-T1W MRI"),
            "NCE-T1W MRI": scan_paths.get("NCE-T1W MRI"),
            "CT-BONE": scan_paths.get("CT-BONE"),
            "CT-BRAIN": scan_paths.get("CT-BRAIN")
        })

# Create master DataFrame
master_df = pd.DataFrame(records)

# Sort records
master_df = master_df.sort_values(
    ["Diagnosis", "Patient ID"]
).reset_index(drop=True)

print("Master dataset shape:", master_df.shape)

display(master_df.head(10))
print("Total patients:", len(master_df))

print("\nPatients by diagnosis:")

display(
    master_df.groupby("Diagnosis")
    .size()
    .reset_index(name="Number of patients")
)

Master dataset shape: (100, 6)


,Diagnosis,Patient ID,CE-T1W MRI,NCE-T1W MRI,CT-BONE,CT-BRAIN
0,meningioma,sub-01,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
1,meningioma,sub-02,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
2,meningioma,sub-03,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
3,meningioma,sub-04,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
4,meningioma,sub-05,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
5,meningioma,sub-06,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
6,meningioma,sub-07,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
7,meningioma,sub-08,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
8,meningioma,sub-09,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...
9,meningioma,sub-10,C:\Users\nihar\Downloads\meningioma\meningioma...,NaN,C:\Users\nihar\Downloads\meningioma\meningioma...,C:\Users\nihar\Downloads\meningioma\meningioma...


Total patients: 100

Patients by diagnosis:


,Diagnosis,Number of patients
0,meningioma,20
1,pituitary tumor,60
2,vestibular schwannoma,20


In [23]:
scan_columns = [
    "CE-T1W MRI",
    "NCE-T1W MRI",
    "CT-BONE",
    "CT-BRAIN"
]

print("Missing scans:")

display(master_df[scan_columns].isnull().sum())

Missing scans:


CE-T1W MRI      0
NCE-T1W MRI    40
CT-BONE         0
CT-BRAIN        0
dtype: int64

In [24]:
import nibabel as nib
import numpy as np

# Inspect the first patient's scans
sample = master_df.iloc[0]

scan_paths = {
    "CE-T1W MRI": sample["CE-T1W MRI"],
    "CT-BONE": sample["CT-BONE"],
    "CT-BRAIN": sample["CT-BRAIN"]
}

for scan_name, file_path in scan_paths.items():

    image = nib.load(file_path)

    print("\nScan:", scan_name)
    print("Image dimensions:", image.shape)
    print("Voxel spacing (mm):", nib.affines.voxel_sizes(image.affine))
    print("Orientation:", nib.aff2axcodes(image.affine))


Scan: CE-T1W MRI
Image dimensions: (440, 440, 381)
Voxel spacing (mm): [0.5 0.5 0.5]
Orientation: ('L', 'A', 'S')

Scan: CT-BONE
Image dimensions: (440, 440, 381)
Voxel spacing (mm): [0.5 0.5 0.5]
Orientation: ('L', 'A', 'S')

Scan: CT-BRAIN
Image dimensions: (440, 440, 381)
Voxel spacing (mm): [0.5 0.5 0.5]
Orientation: ('L', 'A', 'S')


In [26]:
import nibabel as nib
import pandas as pd

image_records = []

for _, row in master_df.iterrows():

    scans = {
        "CE-T1W MRI": row["CE-T1W MRI"],
        "CT-BONE": row["CT-BONE"],
        "CT-BRAIN": row["CT-BRAIN"]
    }

    for scan_name, file_path in scans.items():

        image = nib.load(file_path)

        image_records.append({
            "Diagnosis": row["Diagnosis"],
            "Patient ID": row["Patient ID"],
            "Scan Type": scan_name,
            "Dimensions": image.shape,
            "Voxel Spacing": tuple(
                nib.affines.voxel_sizes(image.affine)
            ),
            "Orientation": nib.aff2axcodes(image.affine)
        })

image_df = pd.DataFrame(image_records)

print("Total image records:", len(image_df))

display(image_df.head(300))

Total image records: 300


,Diagnosis,Patient ID,Scan Type,Dimensions,Voxel Spacing,Orientation
0,meningioma,sub-01,CE-T1W MRI,"(440, 440, 381)","(0.5, 0.5, 0.5)","(L, A, S)"
1,meningioma,sub-01,CT-BONE,"(440, 440, 381)","(0.5, 0.5, 0.5)","(L, A, S)"
2,meningioma,sub-01,CT-BRAIN,"(440, 440, 381)","(0.5, 0.5, 0.5)","(L, A, S)"
3,meningioma,sub-02,CE-T1W MRI,"(440, 440, 381)","(0.5, 0.5, 0.5)","(L, A, S)"
4,meningioma,sub-02,CT-BONE,"(440, 440, 381)","(0.5, 0.5, 0.5)","(L, A, S)"
...,...,...,...,...,...,...
295,vestibular schwannoma,sub-19,CT-BONE,"(378, 378, 321)","(0.5, 0.5, 0.5)","(L, P, S)"
296,vestibular schwannoma,sub-19,CT-BRAIN,"(378, 378, 321)","(0.5, 0.5, 0.5)","(L, P, S)"
297,vestibular schwannoma,sub-20,CE-T1W MRI,"(440, 440, 361)","(0.5, 0.5, 0.5)","(L, P, S)"
298,vestibular schwannoma,sub-20,CT-BONE,"(440, 440, 361)","(0.5, 0.5, 0.5)","(L, P, S)"


In [27]:
# Keep only our selected modalities

selected_image_df = image_df[
    image_df["Scan Type"].isin(
        ["CE-T1W MRI", "CT-BRAIN"]
    )
].copy()

print("Selected image records:", len(selected_image_df))

print("\nImage dimensions:")
display(
    selected_image_df["Dimensions"]
    .value_counts()
    .reset_index()
)

print("\nVoxel spacing:")
display(
    selected_image_df["Voxel Spacing"]
    .value_counts()
    .reset_index()
)

print("\nOrientation:")
display(
    selected_image_df["Orientation"]
    .value_counts()
    .reset_index()
)

Selected image records: 200

Image dimensions:


,Dimensions,count
0,"(440, 440, 381)",10
1,"(439, 439, 381)",10
2,"(439, 439, 361)",8
3,"(440, 440, 341)",6
4,"(414, 414, 321)",6
...,...,...
74,"(500, 500, 299)",2
75,"(477, 477, 341)",2
76,"(444, 444, 381)",2
77,"(378, 378, 321)",2



Voxel spacing:


,Voxel Spacing,count
0,"(0.5, 0.5, 0.5)",198
1,"(0.4300000071525574, 0.4300000071525574, 0.5)",2



Orientation:


,Orientation,count
0,"(L, A, S)",136
1,"(L, P, S)",56
2,"(R, A, S)",6
3,"(L, A, I)",2


In [33]:
spacing_check = selected_image_df.pivot(
    index=["Diagnosis", "Patient ID"],
    columns="Scan Type",
    values="Voxel Spacing"
).reset_index()

spacing_check["Spacing Match"] = (
    spacing_check["CE-T1W MRI"] ==
    spacing_check["CT-BRAIN"]
)

print("Patients with matching voxel spacing:",
      spacing_check["Spacing Match"].sum())

print("Patients with different voxel spacing:",
      (~spacing_check["Spacing Match"]).sum())



Patients with matching voxel spacing: 100
Patients with different voxel spacing: 0


In [35]:
master_df["Unique Patient ID"] = (
    master_df["Diagnosis"] + "_" + master_df["Patient ID"]
)

print("Total unique patients:", master_df["Unique Patient ID"].nunique())

display(
    master_df[
        ["Diagnosis", "Patient ID", "Unique Patient ID"]
    ].head(10)
)

Total unique patients: 100


,Diagnosis,Patient ID,Unique Patient ID
0,meningioma,sub-01,meningioma_sub-01
1,meningioma,sub-02,meningioma_sub-02
2,meningioma,sub-03,meningioma_sub-03
3,meningioma,sub-04,meningioma_sub-04
4,meningioma,sub-05,meningioma_sub-05
5,meningioma,sub-06,meningioma_sub-06
6,meningioma,sub-07,meningioma_sub-07
7,meningioma,sub-08,meningioma_sub-08
8,meningioma,sub-09,meningioma_sub-09
9,meningioma,sub-10,meningioma_sub-10


In [37]:
import numpy as np
import nibabel as nib
import pandas as pd

quality_records = []

for _, row in master_df.iterrows():

    scans = {
        "CE-T1W MRI": row["CE-T1W MRI"],
        "CT-BRAIN": row["CT-BRAIN"]
    }

    for scan_name, file_path in scans.items():

        image = nib.load(file_path)
        data = image.get_fdata()

        quality_records.append({
            "Diagnosis": row["Diagnosis"],
            "Patient ID": row["Unique Patient ID"],
            "Scan Type": scan_name,
            "Min": np.min(data),
            "Max": np.max(data),
            "Mean": np.mean(data),
            "Std": np.std(data),
            "NaN": np.isnan(data).any(),
            "Infinity": np.isinf(data).any(),
            "All Zero": np.all(data == 0)
        })

quality_df = pd.DataFrame(quality_records)

print("Total images checked:", len(quality_df))

print("\nNaN images:", quality_df["NaN"].sum())
print("Infinite images:", quality_df["Infinity"].sum())
print("All-zero images:", quality_df["All Zero"].sum())

Total images checked: 200

NaN images: 0
Infinite images: 0
All-zero images: 0


# Exploratory analysis
*FPR: Methodology; Quality and Results*

In [ ]:
# plots and summary statistics; save useful figures for the report, e.g.
# fig, ax = plt.subplots(); ...; fig.savefig(FIG_DIR / "eda_class_balance.png", dpi=300, bbox_inches="tight")

*What did you learn from the data, and how does it affect your method?*

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: exploratory analysis and first figures"
git push
```

# Method and artefact
*FPR: Methodology*

**Why this method?** Compare with alternatives from the literature (use your extraction sheet).

### Baseline
A simple method to compare against. Build it on its own branch: `git checkout -b baseline`

In [ ]:
# baseline

### Proposed method / artefact
Build it on its own branch: `git checkout -b proposed-method`

In [ ]:
# your main method or artefact (keep functions small and well named, you may be asked to change them live in the demo)

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "CODE: baseline and proposed method"
git push
```

# Experiments
*FPR: Quality and Results*


Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "EXP: E01 baseline, accuracy 0.81"
git push
```

# Testing and validation
*FPR: Methodology (test strategy and validation)*

- **Testing:** check that your code does what you expect (unit tests, edge cases).
- **Validation:** check that your results are reliable (cross-validation, held-out test set, benchmark comparison).

In [ ]:
# simple unit tests, e.g.
# assert clean(df).isna().sum().sum() == 0, "cleaned data should have no missing values"
# assert model.predict(X_test).shape[0] == len(X_test)

In [ ]:
# validation, e.g. k-fold cross-validation or a held-out test set

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "TEST: unit tests and 5-fold cross-validation"
git push
```

# Results
*FPR: Quality and Results*

Save every figure and table you use in the report, then upload them to Overleaf.

In [ ]:
# results = pd.read_csv(RESULTS_DIR / "experiments.csv"); display(results)
# results.to_latex(FIG_DIR / "results_table.tex", index=False)   # paste or \input into Overleaf

*What do the results mean? Link them to your objectives and the literature.*

Save your work, then run these commands in the terminal:
```bash
git status
git add .
git commit -m "RESULT: final results table and figures for FPR"
git push
```

# Demo / viva preparation (20%)
About 10 minutes of demo, then about 10 minutes of questions. Focus on the **practical work**: no slides, no reading the report.

**Plan what to show:**
1. Problem and objective (1 min)
2. Data and how you prepared it
3. Your artefact running: run the cell below from a fresh kernel
4. Key results and what they mean

**Practise these questions:**
- What did you do, why, and how? How did you know how to do it?
- What challenges did you meet, and how did you solve them?
- How did your project change over time? (use your commit history)
- Explain this function and change a parameter live. What happens?
- Where could your code be optimised? Show the change.

In [ ]:
# DEMO: run the whole pipeline end to end from a fresh kernel (Kernel -> Restart and Run All)
# 1. load data  2. clean  3. train / build  4. evaluate  5. show key figure

# Create your release (before you submit the IPR and FPR)
A **release** is a named snapshot of your work at submission. Your supervisor can open exactly what you submitted, with your PDF attached.

| Milestone | Release name |
|---|---|
| Interim Progress Report (10%) | `IPR-v1.0` |
| Final Project Report (70%) | `FPR-v1.0` |
| Optional: fixes before the demo | `Demo-v1.0` |

**Step 1. Save and push your final work**
```bash
git status
git add .
git commit -m "FPR: final version for submission"
git push
```

**Step 2. Create the release on GitHub**
1. On your repository page, click **Releases** (right-hand side) → **Draft a new release**.
2. Click **Choose a tag**, type `FPR-v1.0` and click **Create new tag**. Target: `main`.
3. **Release title:** *Final Project Report*.
4. **Description** (3–5 lines), for example:
   > Final submission. Includes the full pipeline in MSC_final_project.ipynb, 12 logged experiments and the FPR.
   > Changes since IPR-v1.0: added the proposed method and cross-validation.
   > Known issue: training on the full dataset takes about 2 hours.
5. **Attach files:** drag in your submitted PDF (and for the FPR, your code `.txt`).
6. Click **Publish release**.

**Step 3. Put the release name on your cover page**
In Overleaf: `\gitrelease{FPR-v1.0}`. Then submit on Canvas as normal (Canvas is the official submission).

**Check:** your release now appears under **Releases** on your repository page. Do not delete or edit it after you submit.